# Neuronal responses to visual stimuli

*Creator: Dr. Aitor Morales-Gregorio*

<img src="monkey_diagram.png" alt="monkey_diagram" height="150"/>
<img src="brain_render.png" alt="brain_render" height="150"/>
<img src="electrode_and_neurons.png" alt="electrode_and_neurons" height="150"/>


We will analyze spiking data, which we spike sorted last week! We will analize data where some monkeys were shown a checkerboard in the screen:

<img src="SNR_task.png" alt="SNR_task" height="350"/>

The data is from [Chen, Morales-Gregorio et al (2022)](https://www.nature.com/articles/s41597-022-01180-1).

Neurons in the visual cortex respond to bright stimuli (among others). If the experiment is done only once it is hard to tell whether the neuron's activity was higher than normal. To solve this the experiment is repeated many times (**trials**), then we can look at the average activity and find whether the neurons respond to stimulus and how strongly.

### Getting started

In [298]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This loads the spike times from last week
# "st" is short for "spike train"
# We will focus only on two out of the three SUAs 
# (the ones with more spikes in them)
st0 = np.load('SUA0_spike_times.npy')
st1 = np.load('SUA1_spike_times.npy')

In [ ]:
# This loads the metadata
# t_stim_on : the time (in seconds) at which the checkerboard appears in the screen
# success : whether the monkey fixated the gaze the whole time (True), or looked away (False)
# t_rew : the time (in seconds) at which the monkey receives a reward for doing the trial right
metadata = pd.read_csv('epochs_L_SNR_250717.csv')

print(metadata)

# Exercises

### Exercise 1: Peri-stimulus spike raster

1. Get the trial onsets from the metadata, keep only successful trials!

2. Extract the trials for both neurons `st0` and `st1`:
    * For each trial onset get the spike times from 400 ms before onset to 800 ms after the onset
    * Subtract the onset time from the spike times, to make them relative to stimulus presentation
    * Make a list (`trial_sts0` and `trial_sts1`) with the spike times of each trial

3. Use the provided code to plot the trial spike trains
    * The blue shaded area marks when the checkerboard was onscreen
    * What can you say about each neuron? When do they usually respond most reliably?

In [300]:
# Get trial onset times
# ... Your code here ..

# Make a list with the responses of one neuron to each trial
# ... Your code here ..

# These should be a list of numpy arrays
trial_sts0 = ...
trial_sts1 = ...

In [ ]:
# Create figure with two axes
fig, axs = plt.subplots(1, 2, figsize=(11, 2.5), sharey=True,
                        gridspec_kw=dict(hspace=0.1, wspace=0.1))

# For each trial plot the spike times
for i, trial_st0, trial_st1 in zip(range(len(onsets)), trial_sts0, trial_sts1):
    axs[0].plot(trial_st0, [i]*len(trial_st0), lw=0, marker='|', color='k', markersize=3)
    axs[1].plot(trial_st1, [i]*len(trial_st1), lw=0, marker='|', color='k', markersize=3)

# Mark when the checkerboard is on the screen
for ax in axs.flatten():
    ax.fill_between([0, 0.4], [ax.get_ylim()[0]]*2, [ax.get_ylim()[1]]*2,
                    color='skyblue', alpha=0.5, lw=0, zorder=-999)

# Annotate labels
axs[0].set_xlabel('Time from stimulus onset (s)')
axs[1].set_xlabel('Time from stimulus onset (s)')
axs[0].set_ylabel('Trials')
axs[0].set_title('SUA 0')
axs[1].set_title('SUA 1')

plt.show()

## Exercise 2: Time discretization

Also known as binning, time is split into discrete chunks with a certain length (binsize). Here we use 10 ms bins. 

Separately for each neuron `st0` and `st1`:

1. Calculate how many spikes per bin there are, adding together all trials.
    * HINT: use the provided `bin_edges` to count the number of occurrences

2. Plot the binned trial response as a barplot by adding code in the indicated spot.
    * HINT: for plotting you need to use `bin_centers` and set an adequate `width`.

3. When is each neuron's response strongest?
    * **You should get something like:\
        SUA 0: 470 ms after onset\
        SUA 1: 70 ms after onset**

In [302]:
# Set a bin size and define bin edges and centers
binsize = 10 / 1000  # In milliseconds
bin_edges = np.arange(-0.4, 0.8+binsize, step=binsize)
bin_centers = bin_edges[:-1] + binsize/2

# Count number of spikes per bin
# ... Your code here ..

In [ ]:
# Create figure with two axes
fig, axs = plt.subplots(2, 2, figsize=(11, 3.5), sharex=True, sharey='row', 
                        height_ratios=[1, 3], gridspec_kw=dict(hspace=0.1, wspace=0.1))

# For each trial plot the spike times
for i, trial_st0, trial_st1 in zip(range(len(onsets)), trial_sts0, trial_sts1):
    axs[1, 0].plot(trial_st0, [i]*len(trial_st0), lw=0, marker='|', color='k', markersize=3)
    axs[1, 1].plot(trial_st1, [i]*len(trial_st1), lw=0, marker='|', color='k', markersize=3)

# Your changes should START here !!!

# You can comment out these text placeholders
axs[0, 0].text(0.2, 0.5, 'HERE GOES YOUR TRIAL HISTOGRAM', ha='center', va='center', fontweight='bold')
axs[0, 1].text(0.2, 0.5, 'HERE GOES YOUR TRIAL HISTOGRAM', ha='center', va='center', fontweight='bold')

# Plot summed trial response as a barplot
# ... Your code here ..

# Your changes should END here !!!

# Mark when the checkerboard is on the screen
for ax in axs.flatten():
    ax.fill_between([0, 0.4], [ax.get_ylim()[0]]*2, [ax.get_ylim()[1]]*2,
                    color='skyblue', alpha=0.5, lw=0, zorder=-999)

# Annotate labels
axs[0, 0].set_ylabel(f'Number of spikes\nin N = {len(onsets)} trials\nwith {int(binsize*1000)} ms bins', rotation=0, ha='right', va='center')
axs[1, 0].set_xlabel('Time from stimulus onset (s)')
axs[1, 1].set_xlabel('Time from stimulus onset (s)')
axs[1, 0].set_ylabel('Trials')
axs[0, 0].set_title('SUA 0')
axs[0, 1].set_title('SUA 1')

plt.show()

In [ ]:
# Calculate at what time point the peak is
# ... Your code here ..

## Exercise 3: Signal-to-Noise Ratio (SNR)

How strong is the response? Neurons are not silent, so maybe the peak in the trial is just due to random noise!

To check this we can measure the SNR, telling us how much higher the peak response is compared to the baseline background activity:

<img src="SNR_explainer.png" alt="SNR_task" height="350"/>

To measure the SNR from the summed trial response:
$$ SNR = max\ \frac{\nu(t) - \mu_{baseline}}{\sigma_{baseline}}$$
where $\nu(t)$ is the summed trial response, $\mu_{baseline}$ is the mean activity in the baseline and $\sigma_{baseline}$ is the standard deviation in the baseline. For the baseline you can take the 400 ms prior to stimulus onset.

You should get  $SNR_{0} \approx 10.85$  and  $SNR_{1} \approx 14.47$

In [ ]:
# Calculate the SNR
# ... Your code here ..

print('SNR SUA 0:', SNR0)
print('SNR SUA 1:', SNR1)

## Exercise 4: Variability across trials

We have so far looked at the average response across trials (or the summed response, qualitatively they are the same with a different y-scale value). It is also interesting to look at the variability between different trials, since that can indicate the neurons are only weakly driven by the task, or suggest they are very strongly driven by it.

One interesting finding is that variability is usually **reduced** after the stimulus onset ([Churchland et al 2010](https://www.nature.com/articles/nn.2501)). 

We can measure variability using the time-varying Fano Factor:
$$ F(t) = \frac{\sigma^2_{trials}(t)}{\mu_{trials}(t)}$$
where $\mu$ and $\sigma^2$ are the mean and variance **across trials** at a given time point $t$.

So the first thing we need is the time discretized activity, not summed over trials as we had before.
1. Calculate the time discretized activity for each trial (use 30 ms bins)
    * This should be a matrix of 67 trials $\times$ 120 time points, i.e. array of shape (67, 120)
    * Plotting is available to check your result

Now you can use the discretized matrices to calculate the Fano factor over time using `discretized_sts0` and `discretized_sts1`.

2. For each time point calculate the Fano factor $F(t)$
    * Plotting is available

In [368]:
# Set a bin size and define bin edges and centers
# Here we increase the bins to 30 ms
# Shorter bins sometimes get mu=0, which leads to undefined F(t)
binsize = 30 / 1000  # In milliseconds
bin_edges = np.arange(-0.4, 0.8+binsize, step=binsize)
bin_centers = bin_edges[:-1] + binsize/2

# Calculate the discretized trial matrices for each trial
# ... Your code here ..

# These should be arrays of shape (67, 120)
discretized_sts0 = ...
discretized_sts1 = ...

In [ ]:
# Create figure with two axes
fig, axs = plt.subplots(1, 2, figsize=(11, 2.5), sharex=True, sharey='row', 
                        gridspec_kw=dict(hspace=0.1, wspace=0.1))

# For each trial plot the spike times
im = axs[0].imshow(discretized_sts0, cmap='Blues', norm=plt.Normalize(0, 2.5), interpolation='none',
                   origin='lower', extent=[-0.4, 0.8, 0, 67], aspect='auto')
plt.colorbar(im, shrink=0.5, extend='max', label='# spikes')
im = axs[1].imshow(discretized_sts1, cmap='Reds', norm=plt.Normalize(0, 2.5), interpolation='none',
                   origin='lower', extent=[-0.4, 0.8, 0, 67], aspect='auto')
plt.colorbar(im, shrink=0.5, extend='max', label='# spikes')

# Mark when the checkerboard is on the screen
for ax in axs.flatten():
    ax.fill_between([0, 0.4], [ax.get_ylim()[0]]*2, [ax.get_ylim()[1]]*2,
                    color='skyblue', alpha=0.5, lw=0, zorder=-999)

# Annotate labels
axs[0].set_xlabel('Time from stimulus onset (s)')
axs[1].set_xlabel('Time from stimulus onset (s)')
axs[0].set_ylabel('Trials')
axs[0].set_title('SUA 0')
axs[1].set_title('SUA 1')

plt.show()

In [ ]:
# Calculate the Fano factor
# ... Your code here ..
F0 = ... 
F1 = ...

In [ ]:
# This code plots the Fano factor for the two neurons
fig = plt.figure()
ax = fig.add_subplot()

# The effect is very noisy, in the Churchland paper they average over many neurons 
plt.plot(bin_centers, F0, color='b', label='SUA 0, raw F(t)', alpha=0.1)
plt.plot(bin_centers, F1, color='r', label='SUA 1, raw F(t)', alpha=0.1)

# For visualization here we also plot the smoothened the curves
k = 6
plt.plot(bin_centers[k//2:-k//2+1], np.convolve(F0, [1/k]*k, mode='valid'), color='b', label='SUA 0, smooth F(t)', alpha=0.8)
plt.plot(bin_centers[k//2:-k//2+1], np.convolve(F1, [1/k]*k, mode='valid'), color='r', label='SUA 1, smooth F(t)', alpha=0.8)

# Mark when the checkerboad is onscreen
plt.fill_between([0, 0.4], [ax.get_ylim()[0]]*2, [ax.get_ylim()[1]]*2,
                 color='skyblue', alpha=0.5, lw=0, zorder=-999)
plt.legend(loc='upper right')

# Further material

Trials are not limited to showing one image stimulus in the screen. 

Other sensory modalities can also be measured with specialized equipment:
  * Motor tasks with dedicated joysticks, levers etc
  * Auditory responses to sounds
  * Somatosensory responses to air puffs, touches etc
  * Spatial location in a large cage
  * ...

The idea remains the same, similar trials are repeated to gain the power of statistics and elucidate what external (behavioral) variables the neurons are correlated to.
  
An example here with how moving bars can elicit a sequential activation of the neurons in the cortex:

<video width="560" controls>
  <source src="RF_animation_Xing.mp4" type="video/mp4">
</video>

Same monkey but different task, video from [Chen et al (2020)](https://www.science.org/doi/10.1126/science.abd7435).

